# Find a fluorescence correction factopr based on Diya data

In [31]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from bokeh.plotting import figure, show, output_notebook
from bokeh.layouts import gridplot
from bokeh.embed import file_html
from IPython.display import HTML, display
from bokeh.resources import CDN, INLINE
import html as html_lib

from bokeh.sphinxext import bokeh_plot

# Works when launched from the repository root or any directory beneath it.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'Yuhang/plate_reader_analysis_diya_kirill_3.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Launch this notebook from within the CcaSolitaiRe repository.')
DATA_DIR = ROOT / 'Yuhang/Data'

## Settings


In [32]:
LIGHTS = ['red', 'green']                 # Or ['green'] / ['red']; light regions of ONE plate
PROBE_CHANNEL = 'GFP 395nm'
SELECTED_SAMPLES = ['D']#, 'O']
SUMMARY_METRICS = ['change_in_fluorescence', 'od600']
PLOTS = ['replicates', 'blanks', 'summary']
Y_RANGES = {'change_in_fluorescence': (-500,500), 'od600': (0,1)}  # e.g. (0, 1000)
BLANK_LABEL_BY_FAMILY = {'D': 'BD', 'O': 'BO'} # blanks for Diluted and Overnight cultures
BLANK_WELLS = {}                         # e.g. {('green', 'BK_D'): ['E11', 'F11']}
EXCLUDED_MEASUREMENTS = {}               # e.g. {('green', 'D7'): [0]}
SAVE_HTML = False
OUTPUT_DIR = ROOT / 'Yuhang/fluor_corr'


## Plate maps

Arrays and labels copied from the Diya script. Light labels derived from the per-well optical-power arrays.

In [33]:
NCOLS, NROWS = 12, 8
CELL_NAMES = {0: 'BD', 1: 'D', 2: 'O', 4: 'BO',
              # other wells used by yuhang but not here
              3: 'whatever',  5:'null'
              }

cell_map = np.array([
        #1      2       3       4       5       6       7       8       9       10      11      12
        [1,	    1,	    1,  	2,  	2,  	2,  	1,  	1,  	1,  	2,  	2,      2],  # Row A
        [1,	    1,	    1,  	2,  	2,  	2,  	1,  	1,  	1,  	2,  	2,      2],  # Row B
        [1,	    1,	    1,  	2,  	2,  	2,  	1,  	1,  	1,  	2,  	2,      2],  # Row C
        [1,	    1,	    1,  	1,  	1,  	1,  	1,  	1,  	1,  	1,  	1,      1],  # Row D
        [1,	    1,	    1,  	1,  	1,  	1,  	1,  	1,  	1,  	1,  	1,      1],  # Row E
        [1,	    1,	    1,  	1,  	1,  	1,  	1,  	1,  	1,  	1,  	1,      1],  # Row F
        [3,	    0,	    3,  	0,  	3,  	3,  	0,  	3,  	0,  	3,  	3,      3],  # Row G
        [4,	    4,	    3,  	5,  	5,  	5,  	5,  	5,  	5,  	3,  	3,      3],  # Row H
])

OPTICAL_POWER = np.array([
    # Channel 0 – Blue (unused, all zeros)
    np.zeros((NROWS, NCOLS)),

    # Channel 1 – Green light
    np.array([
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row A
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row B
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row C
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row D
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row E
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row F
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row G
    [0.0,	0.0,	0.0,	0.0,	0.0,	0.0,	0.28,	0.28,	0.28,	0.28,	0.28,	0.28],  # Row H
]),

    # Channel 2 – Yellow-Green / White (unused, all zeros)
    np.zeros((NROWS, NCOLS)),

    # Channel 3 – Red light
    np.array([
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row A
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row B
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row C
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row D
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row E
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row F
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row G
    [2.8,	2.8,	2.8,	2.8,	2.8,	2.8,	0.0,	0.0,	0.0,	0.0,	0.0,	0.0],  # Row H
]),
])

Gather all the information into a single plate_map dataframe

In [34]:
# get well labels and sample identities
plate_map = pd.DataFrame([
    {'well': f'{row}{c+1}', 'sample': CELL_NAMES[int(cell_map[r,c])],
     'row': row, 'column': c+1}
    for r, row in enumerate('ABCDEFGH') for c in range(12)
]).set_index('well')

# verify that the plate map has a unique index
assert len(plate_map) == 96 and plate_map.index.is_unique

# get green and red light intensities
plate_map['green_intensity'] = OPTICAL_POWER[1].reshape(-1)
plate_map['red_intensity'] = OPTICAL_POWER[3].reshape(-1)
# verify that all wells are lit EITHER red or green
red_only = plate_map.red_intensity.gt(0) & plate_map.green_intensity.eq(0)
green_only = plate_map.green_intensity.gt(0) & plate_map.red_intensity.eq(0)
if not (red_only | green_only).all():
    raise ValueError('Expected exclusively red or green illumination for every Diya well.')
plate_map['light'] = np.where(green_only, 'green', 'red')

# verify that all the selected samples are included in the plate map
unknown = set(SELECTED_SAMPLES) - set(plate_map['sample'])
if unknown:
    raise ValueError(f'Samples absent from this plate map: {unknown}')
def sample_family(sample):
    return sample.rsplit('_', 1)[0] if sample.rsplit('_', 1)[-1].isdigit() else sample

# get blanks
## match each sample to the corresponding blanks
sample_blank = {}
for sample in SELECTED_SAMPLES:
    family = sample_family(sample)
    if family not in BLANK_LABEL_BY_FAMILY:
        raise ValueError(f'Assign a blank control in BLANK_LABEL_BY_FAMILY for {family}')
    sample_blank[sample] = BLANK_LABEL_BY_FAMILY[family]
blank_labels = list(dict.fromkeys(sample_blank.values()))
## record blanks - making no distinction for light condition this time
blank_wells = {}
for light in LIGHTS:
    if light not in ('red', 'green'):
        raise ValueError(f'Unknown light: {light}')
    blank_wells[light] = {}
    for label in blank_labels:
        wells = list(BLANK_WELLS.get((light, label),
                     plate_map.index[plate_map['sample'].eq(label)].tolist())) # note the absence of light check - all conditions included
        if not wells or len(set(wells)) != len(wells):
            raise ValueError(f'Blank wells for {light}, {label} must be nonempty and unique.')
        if not set(wells) <= set(plate_map.index[plate_map['sample'].eq(label)]):  # note the absence of light check - all conditions included
            raise ValueError(f'Blank wells for {label} must match the control label in the map.')
        blank_wells[light][label] = wells

# verify that each sample has been tested in each light condition
def sample_indices(sample, light):
    return np.flatnonzero(plate_map['sample'].eq(sample) & plate_map['light'].eq(light))
for light in LIGHTS:
    for sample in SELECTED_SAMPLES:
        if not len(sample_indices(sample, light)):
            raise ValueError(f'No wells for {sample}, {light}')

# show the plate map for verification
display(plate_map.pivot(index='row', columns='column', values='sample'))
display(plate_map.pivot(index='row', columns='column', values='light'))

# show how samples are matched to their blanks
for light, blanks in blank_wells.items():
    for label, wells in blanks.items():
        print(f'{light}: {label} blank wells: {", ".join(wells)}')


column,1,2,3,4,5,6,7,8,9,10,11,12
row,,,,,,,,,,,,
A,D,D,D,O,O,O,D,D,D,O,O,O
B,D,D,D,O,O,O,D,D,D,O,O,O
C,D,D,D,O,O,O,D,D,D,O,O,O
D,D,D,D,D,D,D,D,D,D,D,D,D
E,D,D,D,D,D,D,D,D,D,D,D,D
F,D,D,D,D,D,D,D,D,D,D,D,D
G,whatever,BD,whatever,BD,whatever,whatever,BD,whatever,BD,whatever,whatever,whatever
H,BO,BO,whatever,null,null,null,null,null,null,whatever,whatever,whatever


column,1,2,3,4,5,6,7,8,9,10,11,12
row,,,,,,,,,,,,
A,red,red,red,red,red,red,green,green,green,green,green,green
B,red,red,red,red,red,red,green,green,green,green,green,green
C,red,red,red,red,red,red,green,green,green,green,green,green
D,red,red,red,red,red,red,green,green,green,green,green,green
E,red,red,red,red,red,red,green,green,green,green,green,green
F,red,red,red,red,red,red,green,green,green,green,green,green
G,red,red,red,red,red,red,green,green,green,green,green,green
H,red,red,red,red,red,red,green,green,green,green,green,green


red: BD blank wells: G2, G4, G7, G9
green: BD blank wells: G2, G4, G7, G9


# Load raw readings

In [35]:
# initialise raw/processed measurements, measurement times, and blank measuremenys
raw, times, blanks, processed = {}, {}, {}, {}
acquisition_rows, qc_rows = [], []

# read measurements by light condition and by observation channel
for light in LIGHTS:
    raw[light] = {}
    for channel in ['OD600', PROBE_CHANNEL]:
        # read the file
        path = DATA_DIR / f'26-09-24 Kirill single plasmid control_diya_extracted_{channel}.csv'
        frame = pd.read_csv(path, index_col=0)

        # verify that there are records for all 96 well plates
        if not frame.index.is_unique or set(frame.index) != set(plate_map.index):
            raise ValueError(f'Expected exactly 96 unique wells in {path}')

        # get measurement times
        frame = frame.loc[plate_map.index].astype(float)
        stamps = pd.to_datetime(frame.columns, dayfirst=True, format='mixed')
        if not stamps.is_monotonic_increasing or not stamps.is_unique:
            raise ValueError(f'Timestamps must be unique and increasing: {path}')

        # veri that measurements are finite
        if not np.isfinite(frame.to_numpy()).all():
            raise ValueError(f'Non-finite measurements in {path}')

        # get raw measuremenet
        raw[light][channel] = frame

        # get the labels for acquired well
        for i, stamp in enumerate(stamps):
            acquisition_rows.append({'light': light, 'channel': channel, 'reading': i, 'timestamp': stamp})

        # it this is an OD measurement, set its time of acquisition for all measured variables
        if channel == 'OD600':
            times[light] = ((stamps - stamps[0]).total_seconds() / 3600).to_numpy()

    # get raw data for all samples
    od_raw = raw[light]['OD600'].to_numpy()
    flu_raw = raw[light][PROBE_CHANNEL].to_numpy()
    if od_raw.shape != flu_raw.shape:
        raise ValueError(f'Different numbers of OD and fluorescence data points for {light}')
    # get mean blank ODs
    blanks[light] = {
        label: {channel: frame.loc[wells].mean(axis=0).to_numpy()
                for channel, frame in raw[light].items()}
        for label, wells in blank_wells[light].items()
    }

# show the labels and time stamps for all acquired data
display(pd.DataFrame(acquisition_rows))

,light,channel,reading,timestamp
0,red,OD600,0,2026-09-23 12:48:08
1,red,OD600,1,2026-09-23 15:50:45
2,red,OD600,2,2026-09-24 00:57:05
3,red,OD600,3,2026-09-24 12:52:18
4,red,OD600,4,2026-09-24 16:33:47
5,red,GFP 395nm,0,2026-09-23 12:49:31
6,red,GFP 395nm,1,2026-09-23 15:52:08
7,red,GFP 395nm,2,2026-09-24 00:58:28
8,red,GFP 395nm,3,2026-09-24 12:53:41
9,red,GFP 395nm,4,2026-09-24 16:35:11


Normalise OD according to the formula:

- `OD = max(raw OD − matched blank mean OD, 0.005)`

Find changes in fluorescence according to the formula:

- `change_fluorescence = raw fluorescence − matched blank mean fluorescence`

In [36]:
for light in LIGHTS:
    for channel in ['OD600', PROBE_CHANNEL]:
        # Normalize measurements for selected samples; blanks/media/unselected samples get nans
        ## initialize with nans
        processed[light] = {metric: np.full_like(od_raw, np.nan)
                            for metric in ['od600', 'change_in_fluorescence']}
        for sample in SELECTED_SAMPLES:
            idx = sample_indices(sample, light)
            label = sample_blank[sample]
            blank = blanks[light][label]
            # normalize the OD and floor it
            od_delta = od_raw[idx] - blank['OD600']
            od = np.maximum(od_delta, 0.005)

            # get the change in fluorescence
            flu_delta = flu_raw[idx] - blank[PROBE_CHANNEL]

            # gather the processed data
            processed[light]['od600'][idx] = od
            processed[light]['change_in_fluorescence'][idx] = flu_delta
            qc_rows.append({'light': light, 'sample': sample, 'blank_control': label,
                            'blank_wells': ','.join(blank_wells[light][label]),
                            'technical_wells': len(idx),
                            'OD_values_floored': int((od_delta < 0.005).sum())})

# show the processed data frame
display(pd.DataFrame(qc_rows))

,light,sample,blank_control,blank_wells,technical_wells,OD_values_floored
0,red,D,BD,"G2,G4,G7,G9",27,7
1,red,D,BD,"G2,G4,G7,G9",27,7
2,green,D,BD,"G2,G4,G7,G9",27,3
3,green,D,BD,"G2,G4,G7,G9",27,3


In [37]:
# Masks are applied to samples, not to the blank averages.
masks = {light: np.ones_like(processed[light]['od600'], dtype=bool) for light in LIGHTS}
for (light, well), indices in EXCLUDED_MEASUREMENTS.items():
    if light not in masks or well not in plate_map.index:
        raise ValueError(f'Unknown mask target: {light}, {well}')
    if plate_map.loc[well, 'light'] != light or plate_map.loc[well, 'sample'] not in SELECTED_SAMPLES:
        raise ValueError(f'Mask target must be a selected sample under {light}: {well}')
    if any(i < 0 or i >= len(times[light]) for i in indices):
        raise ValueError(f'Invalid reading index for {light}, {well}')
    masks[light][plate_map.index.get_loc(well), indices] = False

# Masks are applied to samples, not to the blank averages.
masks = {light: np.ones_like(processed[light]['od600'], dtype=bool) for light in LIGHTS}
for (light, well), indices in EXCLUDED_MEASUREMENTS.items():
    if light not in masks or well not in plate_map.index:
        raise ValueError(f'Unknown mask target: {light}, {well}')
    if plate_map.loc[well, 'light'] != light or plate_map.loc[well, 'sample'] not in SELECTED_SAMPLES:
        raise ValueError(f'Mask target must be a selected sample under {light}: {well}')
    if any(i < 0 or i >= len(times[light]) for i in indices):
        raise ValueError(f'Invalid reading index for {light}, {well}')
    masks[light][plate_map.index.get_loc(well), indices] = False

# Gather the data into a summary dataframe
summary_rows = []
for light in LIGHTS:
    for sample in SELECTED_SAMPLES:
        idx = sample_indices(sample, light)
        for metric, values in processed[light].items():
            for t, hours in enumerate(times[light]):
                included = values[idx, t][masks[light][idx, t]]
                summary_rows.append(dict(light=light, sample=sample, metric=metric,
                                         blank_control=sample_blank[sample], reading=t, hours=hours, n=len(included),
                                         mean=included.mean() if len(included) else np.nan,
                                         sd=included.std(ddof=0) if len(included) else np.nan))
summary = pd.DataFrame(summary_rows)
# show a representatie entry in the summary dataframe
display(summary.head(12))

,light,sample,metric,blank_control,reading,hours,n,mean,sd
0,red,D,od600,BD,0,0.000000,27,0.014300,0.008122
1,red,D,od600,BD,1,3.043611,27,0.415848,0.030122
2,red,D,od600,BD,2,12.149167,27,0.542129,0.049715
3,red,D,od600,BD,3,24.069444,27,0.651884,0.101586
4,red,D,od600,BD,4,27.760833,27,0.694618,0.044738
5,red,D,change_in_fluorescence,BD,0,0.000000,27,-23.064815,63.593056
6,red,D,change_in_fluorescence,BD,1,3.043611,27,-276.750000,45.110564
7,red,D,change_in_fluorescence,BD,2,12.149167,27,-224.564815,36.262456
8,red,D,change_in_fluorescence,BD,3,24.069444,27,-195.305556,123.057148
9,red,D,change_in_fluorescence,BD,4,27.760833,27,-144.462963,40.661640


## Plots

Replicate panels show each technical well plus its masked mean. Blank panels show raw selected blank wells and their mean, before OD flooring. Summary panels group `K_D`, `N_D`, etc., retain a separate curve for each biological replicate, and show ±1 SD across its technical wells. Click legend entries to hide curves. Bokeh's toolbar supports zoom, pan, reset and image saving; `SAVE_HTML` also writes interactive standalone HTML files.

In [38]:
# specify plotting styles
COLORS = {'red': 'crimson', 'green': 'darkgreen'}
DASHES = ['solid', 'dashed', 'dotted']
LABELS = {'od600': 'Blanked OD600',
          'change_in_fluorescence': f'{PROBE_CHANNEL} change from blank, AU'}

# figure creation function
def new_figure(title, ylabel, metric=None):
    options = {} if Y_RANGES.get(metric) is None else {'y_range': Y_RANGES[metric]}
    return figure(title=title, x_axis_label='Time since first OD reading, h',
                  y_axis_label=ylabel, width=400, height=300,
                  tools='pan,wheel_zoom,box_zoom,reset,save', **options)

#  show all plots after styling the legends
def emit(layout, name):
    from bokeh.models import Legend
    for legend in layout.select({'type': Legend}):
        legend.location = 'top_left'
        legend.background_fill_color = 'white'
        legend.background_fill_alpha = 1.0
        legend.border_line_color = 'black'
        legend.border_line_alpha = 1.0
    html_doc = file_html(layout, CDN, 'Plots')
    display(HTML(f'<iframe srcdoc="{html_lib.escape(html_doc)}" width="100%" height="700" style="border:none;"></iframe>'))
    if SAVE_HTML:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        channel_slug = PROBE_CHANNEL.replace(' ', '_')
        target = OUTPUT_DIR / f'{channel_slug}_{name}.html'
        target.write_text(file_html(layout, INLINE, name), encoding='utf-8')
        print(f'Saved {target}')

# plot data for different replicates
if 'replicates' in PLOTS:
    for metric in SUMMARY_METRICS:
        panels = []
        for sample in SELECTED_SAMPLES:
            for light in LIGHTS:
                p = new_figure(f'{sample}, {light}', LABELS[metric], metric)
                idx = sample_indices(sample, light)
                for j, row in enumerate(idx):
                    values = np.where(masks[light][row], processed[light][metric][row], np.nan)
                    p.line(times[light], values, color=COLORS[light], alpha=0.35,
                           line_dash=DASHES[j % 3], legend_label=plate_map.index[row])
                s = summary.query('light == @light and sample == @sample and metric == @metric')
                p.line(s.hours, s['mean'], color=COLORS[light], line_width=3, legend_label='Mean')
                p.legend.padding = 0
                p.legend.margin = 2
                p.legend.spacing = 0
                p.legend.click_policy = 'hide'
                panels.append(p)
        if panels:
            emit(gridplot(panels, ncols=len(LIGHTS)), f'replicates_{metric}')

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Plot blanks:

In [39]:
if 'blanks' in PLOTS:
    panels = []
    for light in LIGHTS:
        for label, wells in blank_wells[light].items():
            for channel in ['OD600', PROBE_CHANNEL]:
                p = new_figure(f'{light}: {label} ({", ".join(wells)})', f'Raw {channel}')
                for well in wells:
                    p.line(times[light], raw[light][channel].loc[well].to_numpy(),
                           color=COLORS[light], alpha=0.35, legend_label=well)
                p.line(times[light], blanks[light][label][channel], color='black',
                       line_width=3, legend_label='Control mean')
                p.legend.padding = 0
                p.legend.margin = 2
                p.legend.spacing = 0
                p.legend.click_policy = 'hide'
                panels.append(p)
    emit(gridplot(panels, ncols=2), 'blanks')

Plot OD vs fluorescence reduction

In [40]:
p = figure(title='OD vs fluorescence reduction', x_axis_label='OD600', y_axis_label=f'{PROBE_CHANNEL} change from blank, AU',
                  width=600, height=450, tools='pan,wheel_zoom,box_zoom,reset,save')
for light in LIGHTS:
    for sample in SELECTED_SAMPLES:
        idx = sample_indices(sample, light)
        od_values = processed[light]['od600'][idx]
        flu_values = processed[light]['change_in_fluorescence'][idx]
        p.circle(od_values.flatten(), flu_values.flatten(),
                 color=COLORS[light], alpha=0.35, legend_label=f'{sample}, {light}', radius=0.01)
p.legend.padding = 0
p.legend.margin = 2
p.legend.spacing = 0
p.legend.click_policy = 'hide'
emit(p, 'od_vs_fluorescence_reduction')

/home/ersat/anaconda3/envs/ucl-wsl-tretayuga/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
